# Lab 2 Agent Pipeline (experiments and retrieval over your saved pipeline)
**Objective:** direct an agent through classification experiments and retrieval comparisons over the pipeline you built and saved in `DM2026-Lab2-Master.ipynb`, and audit what it reports.

This notebook comes after the Master. There you built five representations of the same tweets, trained classifiers on them, and measured retrieval; the Master's Section 9 saved all of it to `lab2_workspace/`. Here an agent runs experiments on that saved pipeline under your direction. It is the same kind of agent as in the Lab 1 agentic notebooks, built on the same `agent_pipeline/` package: one tool call per message, a mandatory plain-text comment after every tool call, an approval step before any tool that computes something new, and a check on every number it states.

Nothing here re-teaches the Master. The agent runs and compares; your part is to decide what is worth running, one controlled step at a time, and to check every claim against the tool output that produced it.

## Table of Contents
0. Setup: your name, a backend, and its API key
1. Load the saved pipeline
2. What the agent can do, and what to check
3. Build the graph and launch the chat
4. Exercises (14 to 16)
5. Submitting your session log

## 0. Setup: your name, a backend, and its API key
**Objective:** identify yourself for the session log and choose the language model that powers the agent.

Enter your name and student ID exactly as they should appear on the submission. Both are written into your session log's plain-text header, so the log can be matched to you however the file is renamed. The backends and the `config/.env` key are the same as in Lab 1 (see `config/.env.example`). **Set up both providers if you can.** Either one answers a tool-calling turn in a few seconds, but a single provider runs out of its per-minute token budget quickly here: every message the agent handles resends the system prompt and all fourteen tool descriptions, so one key alone stalls after a couple of messages. With both configured and fallback on, seven messages in a row go through with no waiting, the chat switches providers when one budget ran out and switches back a minute later once it recovered.

**A second provider helps more than a second key of the same kind.** Groq applies its rate limit to your whole account, so extra Groq keys add no quota. Extra Gemini keys help only when each comes from a separate Google project, since each project has its own limit. One key from each provider is the combination that keeps this notebook moving.

If you have keys for both providers, **enable multi-provider fallback**. When a backend hits a rate limit, or its servers are unreachable or overloaded, the agent retries on another key or provider instead of stopping, and the chat prints a line whenever it switches. The limit you will actually meet is a per-minute one, and it clears within about a minute. The daily quotas are much larger and reset once a day (Groq at midnight UTC, Gemini at midnight Pacific time). It's recommended to start with time to spare, just in case.

You can type `/status` in the chat at any point to see which of your keys are still active, which have hit a rate limit this session, and how large the requests are getting. The chat widget answers that itself: it never reaches the agent, costs no turn, and is not written to your session log. `/keys` does the same thing.

Run the cell below, fill it in, and click **Confirm** before moving on.

In [ ]:
from agent_pipeline.backend_widget import select_backend

backend_choice = select_backend()

## 1. Load the saved pipeline
**Objective:** load the agent's tools and the saved pipeline from disk, so this notebook starts from a fresh kernel and still works on exactly what the Master built.

The tools come from two folders. `lab2_tools/inspection/` holds tools that only read what is already saved (the list of representations, the Master's results table, breakdowns of earlier results). They compute nothing new and send no tweets to the model, so they are usable immediately. `lab2_tools/experiments/` holds tools that train a classifier, run retrieval, or show example tweets; those need your explicit approval before they run, through the same request and approval exchange as Lab 1's agentic pipeline, and one approval covers all five of them at once rather than one tool at a time.

Every tool reads `lab2_workspace/`, never this kernel's memory. That is why the agent's numbers can be checked against the Master: a run with the Master's settings reproduces the Master's score to within a prediction or two. If the cell below reports that the workspace is missing, run `DM2026-Lab2-Master.ipynb` through Section 9 first.

In [ ]:
import os

from agent_pipeline.session_state import SessionState
from agent_pipeline.workspace_loader import load_workspace_tools
from agent_pipeline.tool_access import build_gated_tools

if not os.path.isfile("lab2_workspace/manifest.json"):
    print("lab2_workspace/ not found. Run DM2026-Lab2-Master.ipynb through Section 9 first, then re-run this cell.")

session = SessionState()
experiment_tools, experiment_problems = load_workspace_tools("lab2_tools/experiments", session)
inspection_tools, inspection_problems = load_workspace_tools("lab2_tools/inspection", session)
tools = build_gated_tools(experiment_tools, session, premade_tools=inspection_tools)
problems = experiment_problems + inspection_problems

print(f"Experiment tools ({len(experiment_tools)}, need your approval to run): {[t.name for t in experiment_tools]}")
print(f"Inspection tools ({len(inspection_tools)}, usable right away): {[t.name for t in inspection_tools]}")
if problems:
    print("\nNot loaded:")
    for fname, message in problems:
        print(f"  {fname}: {message}")

## 2. What the agent can do, and what to check
**Objective:** know the tools well enough to direct them precisely, and know what to verify in the agent's replies before trusting them.

| Tool | Needs approval | What it does |
|---|---|---|
| `list_representations_tool` | no | Lists the five saved representations and the dataset's schema (column names and types, row and label counts). |
| `show_master_results_tool` | no | The Master's classification results table, best first: the baseline to compare against. |
| `inspect_result_tool` | no | Per-emotion precision, recall, and F1, the confusion matrix, and the most frequent confusion for one classifier run. |
| `compare_results_tool` | no | Two or more classifier runs side by side, with the gap between best and worst. |
| `run_classifier_tool` | yes | Trains KNN, a linear SVM, Naive Bayes, or an MLP on one representation and evaluates it on the test tweets. |
| `show_misclassified_tool` | yes | Up to 10 test tweets one run got wrong, with true and predicted emotion and the annotated intensity. |
| `retrieve_tool` | yes | The top *k* training tweets for a query under one retriever (TF-IDF, Word2Vec, or MiniLM). |
| `compare_retrieval_tool` | yes | One query through all three retrievers, with how many documents each pair shares. |
| `evaluate_retrieval_tool` | yes | Label precision@*k* for one retriever over all test queries, with the random baseline. |

The agent also has the five access tools you know from Lab 1 (list, describe, request, grant, discard).

**The first experiment tool you ask for takes three messages of your own, and it does not run until the third.** The graph allows the agent one tool call per message, and granting access is itself a tool call, so approving and running cannot happen together. The approval is a batch: it covers all five experiment tools at once, so this exchange happens once per session, not once per tool:

| Your message | What the agent does | What you should see |
|---|---|---|
| 1. Ask for the step, with the details specified | Calls `request_tool_access_tool` and explains why | A pending request. No result yet. |
| 2. Approve it (for example "yes, go ahead") | Calls `grant_tool_access_tool` | A confirmation that all five experiment tools are now usable. **Still no result:** nothing has run yet. |
| 3. Ask for the run again | Calls the tool itself | The result, with a `result_id` |

After that, every experiment tool runs directly for the rest of the session, one call per message, including the ones you have not asked for yet. The agent will not request and grant in one step, and it will not chain two steps into one message, however the request is phrased.

**If approving seems to do nothing.** Approving produces a confirmation, not a result, so message 2 always looks quiet. When the agent answers with nothing but the tool call it wants next, the chat prints that step instead of a blank reply, as `(Done with that step. Next it wants to run ... Say "go ahead" to run it, or ask for something else.)`, which is your cue to send the next message. If the agent replies with a comment but never calls `grant_tool_access_tool`, repeat the approval. If it tries to run the tool in the same message as the grant, the run is refused (it was not granted yet) and the grant is dropped, so nothing changes at all: approve on its own, and ask for the run in a separate message.

**What to check in every reply.** The agent's prose is a claim about tool output, not the output itself. The tables in the chat come directly from the tool; the sentences around them come from the model. Each time, check:
- that every number it states appears in the tool output above it (the chat marks any number it cannot trace, and the mark is also recorded in your log);
- that it describes the tweets actually shown, not tweets it imagines (compare its summary with the table);
- that a claimed improvement is larger than the differences noise alone would produce; on 347 test tweets, a gap in the third decimal is not a finding;
- that a run with the Master's settings reports a `master_reference_macro_f1` within a prediction or two of its own `macro_f1`, and that you find out why if the gap is wider.

**What your log records.** Each message you send is logged with the tool call it triggered (name and arguments), the tool's result, and the agent's reply, encrypted as it is written. Grading reads that sequence: specific, one-step requests that build on earlier results score well; vague requests, requests copied from this notebook, and approvals given without reading what was proposed do not.

**If the agent's reply is only "(No response came back for that...)".** The tool call already ran and its result is shown above; only the model's comment on it came back empty. Ask it to summarize the result, or continue with your next request. Nothing needs restarting.

**If replies seem to be missing entirely, check your widget versions.** With `ipywidgets` 8.x the chat's input row renders inside the scrollable conversation box, so new replies can appear underneath it and look absent. From the repository root, run `uv sync --locked --python 3.11` to restore the pinned `ipywidgets==7.8.1` environment, then restart Jupyter with `uv run --locked jupyter lab`.

**If the chat does not appear at all, restart JupyterLab itself, not just the kernel.** Now and then the launch cell renders an error or nothing where the chat should be, and restarting the kernel does not clear it, because the stale widget state lives in the browser tab and the running server rather than in the kernel. Close the JupyterLab tab, stop the server in the terminal you started it from (`Ctrl+C`), start it again with `uv run --locked jupyter lab`, and run the notebook from the top. The chat renders normally after that, and nothing you have already logged is lost.

**If the chat reports a rate limit or an unreachable provider, or tells you to wait and send another message.** Every configured key and provider failed in a row, and the chat handed control back to you rather than freezing the notebook. Do not restart or stop the kernel: the retry progress lives only in its memory, and a restart starts the count over. Wait a few minutes and send your next message; the retries continue where they stopped. Type `/status` to see which keys are still active before you spend a turn on a real request.


## 3. Build the graph and launch the chat
**Objective:** connect the tools, the rules, and your chosen model into the agent, and open the chat.

The graph is the one Lab 1's agentic pipeline uses (`agent_pipeline/graph_core.py`): one tool call per message, then a forced plain-text comment on the result, then a stop to wait for you. The system prompt at the top of the cell states the rules the agent works under; read it once, since it defines what you can reasonably ask for. Re-running this cell starts a fresh conversation, and the log keeps the earlier one.

In [ ]:
SYSTEM_PROMPT = """You run experiments over the student's saved Lab 2 pipeline. \
DM2026-Lab2-Master.ipynb saved it to lab2_workspace/: five representations \
(tfidf, word2vec, doc2vec, fasttext, minilm) of the same SemEval-2017 emotion \
tweets (anger, fear, joy, sadness), the fitted models, and the Master's own \
results table. Your tools read those files. You never see the dataset itself, \
only small summaries and, when the student asks, at most ten example tweets.

Inspection tools (list_representations_tool, show_master_results_tool, \
inspect_result_tool, compare_results_tool) only read saved results and are \
usable immediately. Experiment tools (run_classifier_tool, \
show_misclassified_tool, retrieve_tool, compare_retrieval_tool, \
evaluate_retrieval_tool) compute new results or send tweets to you, and need \
the student's approval first, granted once as a batch covering all five. \
Calling one before that returns a refusal, not a result. Where the access \
tools say "the student's own tools", they mean these five.

Rules you must follow:
1. Call AT MOST ONE tool per response, and only when the student has asked for \
a specific action with the details it needs.
2. If a request is ambiguous (which representation, classifier, setting or \
query), ask instead of guessing.
3. While the batch is ungranted, call request_tool_access_tool with a reason \
tied to what the student asked, then stop and wait. It takes only that reason \
and proposes all five at once. When the student's next message approves, call \
grant_tool_access_tool as that turn's ONLY tool call: never in the same turn \
as the request, and never in the same turn as the tool being granted, since \
only your first tool call in a turn runs. Never answer an approval with plain \
text alone; make the grant call. That one grant makes all five usable, so say \
so and name the exact request to send next. Once granted, never call \
request_tool_access_tool again. If the student declines, call \
discard_tool_request_tool.
4. Answer what tools exist or what one does with list_available_tools_tool or \
describe_tool_tool, never from a name alone.
5. After a tool returns, report what it shows, quoting the actual values and \
doc_ids rather than paraphrasing them.
6. Every number you state must come from a tool's returned data. Do not \
compute, estimate or invent one, and do not call a difference an improvement \
without stating its size.
7. Refuse to chain several steps in one message; the student directs each step.
8. Refer to an earlier result by its exact result_id.
9. Right after a tool result you get exactly one turn to comment in plain \
text, with no tool call of any kind. That applies only to that turn; the \
student's next message allows a tool call again.
"""

from agent_pipeline.graph_core import build_agent_graph
from agent_pipeline.chat_widget import launch_chat

if "backend" not in backend_choice:
    raise RuntimeError("Pick a backend in the setup cell above and click Confirm before running this one.")
LLM_BACKEND = backend_choice["backend"]
MULTI_PROVIDER_MODE = backend_choice["multi_provider"]
STUDENT_NAME = backend_choice["student_name"]
STUDENT_ID = backend_choice["student_id"]

if MULTI_PROVIDER_MODE:
    from agent_pipeline.llm_fallback import build_fallback_llm
    llm, active_chain = build_fallback_llm(LLM_BACKEND, temperature=0)
    print(f"Fallback chain (in order): {active_chain}")
else:
    from agent_pipeline.llm_backends import build_llm_for_backend
    llm = build_llm_for_backend(LLM_BACKEND, temperature=0)

app = build_agent_graph(tools, SYSTEM_PROMPT, llm, session)

graph_state, logger = launch_chat(
    app, session, log_path="session_logs/lab2_agent_pipeline_session.jsonl.enc",
    student_name=STUDENT_NAME, student_id=STUDENT_ID, track="lab2-agent-pipeline",
    placeholder="Ask the agent to run one experiment... (or type /status to see key rotation status)",
    llm=llm,
)

## 4. Exercises
**Objective:** practice directing controlled experiments through the agent and auditing its reports against tool output.

Answer each exercise in the cell below it, citing the `result_id`s from your conversation. The graded evidence is what you write here together with the logged conversation that produced it.

### **>>> Exercise 14 (take home):**
**Purpose:** beat a number the Master already obtained, and claim no more than the margin allows.

Pick any row of the Master's results table and try to beat its macro F1. Reproduce it first (same representation), classifier and settings, and check your `macro_f1` against the `master_reference_macro_f1` beside it. That is your baseline. Then make your attempts, one run per message. Ask the agent what `run_classifier_tool` can vary, and count the table: seventeen rows, but five representations times four classifiers is twenty. Four or five runs is enough; finish with `compare_results_tool`.

In the cell below, give your baseline and best attempt with their `result_id`s and say whether you beat the Master. The Master read 0.006 macro F1 as three of the 347 test predictions (Section 7); convert your margin the same way and say whether it is a finding or a coin flip. **If nothing beat the Master's setting, say so and show the runs**. Close with two sentences: one quoting an agent claim you checked yourself, and one on whether your best score is a fair estimate for new tweets, given that you chose it on the test set.

*What to check in the agent's output:* each run changed only what you asked, and its summary of the `compare_results_tool` table matches the table.

In [ ]:
#Answer here

### **>>> Exercise 15 (take home):**
**Purpose:** judge what three representations retrieve for the same query, and weigh one query's verdict against an average over 347.

1. Write two queries of your own about a topic this dataset contains. Predict which of the three retrievers will serve each better, and report where you were wrong.
2. Have the agent run `compare_retrieval_tool` on each, one per message. Read the three tables of retrieved tweets yourself before reading the agent's summary.
3. Have it run `evaluate_retrieval_tool` for TF-IDF and MiniLM at the same *k*. It repeats the Master's Section 8.2 computation, so check its numbers against the table you printed there.

In the cell below, for each query: which retriever's list would you use, citing `doc_id`s, and does the agent's summary of it match the table? What does Word2Vec's list say about that representation? Your queries may make one retriever look far better than another while precision@*k* separates them by a few hundredths: explain how both can be true.

*What to check in the agent's output:* the overlap counts it quotes match the `overlap` table; it does not present one query's result as a general conclusion.

In [ ]:
#Answer here

### **>>> Exercise 16 (take home):**
**Purpose:** treat an agent's explanation of model errors as a set of claims to test against evidence.

In the Master's Exercise 8 you explained your best model's most frequent confusion from the confusion matrix alone, yet never saw a misclassified tweet. 

One request per message: have the agent run a classifier (Exercise 14's best, or the Master's TF-IDF linear SVM) so there is a `result_id`, break it down with `inspect_result_tool`, then show up to five tweets from one confusion with `show_misclassified_tool` and explain why the model got them wrong. Pick the confusion from the whole matrix yourself: `most_confused` names one direction, where Exercise 8 had you add up both.

In the cell below, say which confusion you chose and why, and put the agent's explanation next to your Exercise 8 paragraph. Which of its sentences say something your tweet-blind paragraph could not have said, and which tweets support them (cite `doc_id`s). For the rest, say what in the table could have shown them wrong.

*What to check in the agent's output:* it describes the tweets in the table and not tweets it invented, and refers to them by `doc_id`.

In [ ]:
#Answer here

## 5. Submitting your session log
**Objective:** confirm the log exists before you submit it.

Your conversation is logged turn by turn to `session_logs/lab2_agent_pipeline_session.jsonl.enc`, encrypted with the course's grading public key as it is written. You cannot read or edit it; upload it exactly as produced, together with this notebook (Exercises 14 to 16 answered) and the Master notebook's log, `session_logs/lab2_main_session.jsonl.enc`.

For the complete lab submission, push your work to your repository and submit its link plus a backup ZIP on NTU COOL by **November 25 at 11:59 pm**. See `README.md` for the ZIP contents.

In [ ]:
import os

log_path = "session_logs/lab2_agent_pipeline_session.jsonl.enc"
if os.path.exists(log_path):
    print(f"Log file present: {log_path} ({os.path.getsize(log_path)} bytes)")
else:
    print("No log yet. Send at least one message in the chat above first.")